# Step 1 — Synthetic Patient Data Generation

This notebook documents the **full process of generating the synthetic training data** used for the Knowledge Distillation pipeline.

## What we're building

We need thousands of realistic diabetic patient glucose logs paired with expert clinical analysis. Since real patient data is private, we:

1. **Define a seed population** — demographic profiles drawn from `master_seeds.json`
2. **Generate synthetic 10-day logs** — glucose readings, meals, insulin doses in a compact log format
3. **Query a frontier model** — send each synthetic patient's log to Claude/GPT-4 with a structured prompt to generate expert annotations
4. **Collect raw batches** — the frontier model responses are saved to `../step2_raw_batches/`

## Compact log format

```
D1 09:00 G=138 | 09:30 M=cereal C=50 I=6R | 11:15 G=250 | 13:00 G=145 | 18:30 G=125 | 22:30 I=28L
D2 08:15 G=130 | 08:45 M=pancake C=55 I=7R | 10:30 G=260 | 12:30 G=140 | 18:00 G=120 | 22:00 I=28L
```

Where:
- `G=` — blood glucose in mg/dL
- `M=` — meal name
- `C=` — carbohydrates in grams
- `I=` — insulin dose (e.g., `6R` = 6 units rapid-acting, `28L` = 28 units long-acting)
- `EX=` — exercise event (e.g., `EX=walk`)

## 1. Explore the Seed Population

`master_seeds.json` contains the base population from which we draw synthetic patient profiles. Each seed defines:
- Demographic attributes (age, sex, weight)
- Diabetes type (Type 1 / Type 2)
- Insulin regimen (MDI, pump, basal-only, oral agents)
- Glycemic baseline and variability parameters

In [ ]:
import json
from collections import Counter

with open('master_seeds.json', 'r', encoding='utf-8') as f:
    seeds = json.load(f)

print(f"Total seeds: {len(seeds)}")
print(f"\nSample seed:")
print(json.dumps(seeds[0], indent=2))

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Analyse the seed distribution
diabetes_types = Counter(s.get('diabetes_type', 'unknown') for s in seeds)
regimens = Counter(s.get('regimen', 'unknown') for s in seeds)
ages = [s.get('age', 0) for s in seeds if 'age' in s]

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle('Seed Population Distribution', fontsize=14, fontweight='bold')

axes[0].bar(diabetes_types.keys(), diabetes_types.values(), color=['#4A90D9', '#E85454'])
axes[0].set_title('Diabetes Type')
axes[0].set_ylabel('Count')

axes[1].bar(range(len(regimens)), regimens.values(), tick_label=list(regimens.keys()))
axes[1].set_title('Insulin Regimen')
axes[1].tick_params(axis='x', rotation=30)

if ages:
    axes[2].hist(ages, bins=20, color='#5BAD6F', edgecolor='white')
    axes[2].set_title('Age Distribution')
    axes[2].set_xlabel('Age (years)')

plt.tight_layout()
plt.show()

print(f"\nDiabetes types: {dict(diabetes_types)}")
print(f"Regimens: {dict(regimens)}")
if ages:
    print(f"Age range: {min(ages)}–{max(ages)}, mean: {np.mean(ages):.1f}")

## 2. Log Generation Logic

The following helper functions show how we generate a synthetic 10-day log for a single patient seed. The log uses the compact format that the frontier model was trained to reason about.

Key design decisions:
- **Realistic timing**: meals at typical hours, glucose readings 4–7 per day
- **Physiological plausibility**: post-meal spikes, insulin effects, dawn phenomenon
- **Diversity**: different meal types, exercise events, occasional hypoglycemia
- **Noise**: some missed boluses, carb mis-counts, occasional skipped readings

In [ ]:
import random
from datetime import datetime, timedelta

# Meal library (subset) — 368 unique meal names appear in the full dataset
MEALS = {
    'breakfast': [
        ('oatmeal', 55), ('eggs_toast', 20), ('cereal', 45), ('yogurt_granola', 35),
        ('pancakes', 65), ('bagel', 50), ('omelet', 10), ('smoothie', 40),
    ],
    'lunch': [
        ('wrap', 45), ('salad', 15), ('soup', 30), ('chicken_rice', 60),
        ('sandwich', 40), ('pasta', 70), ('tuna_salad', 10), ('burger', 50),
    ],
    'dinner': [
        ('salmon_veggies', 20), ('steak_veggies', 15), ('pasta_marinara', 75),
        ('pizza', 80), ('chicken_kale', 10), ('fish_rice', 55), ('tacos', 50),
    ],
    'snack': [
        ('apple', 25), ('crackers', 20), ('glucose_tabs', 15), ('juice', 30),
        ('nuts', 8), ('yogurt', 20), ('candy', 15), ('banana', 27),
    ]
}

EXERCISES = ['walk', 'run', 'bike', 'swim', 'yoga', 'weights', 'jog']


def generate_daily_log(seed: dict, day: int, prev_glucose: int) -> tuple[str, int]:
    """Generate a single day log line in compact format.
    Returns (log_string, ending_glucose).
    """
    rng = random.Random(seed.get('seed_id', 0) * 1000 + day)
    
    diabetes_type = seed.get('diabetes_type', 'Type 2')
    uses_insulin = seed.get('regimen', 'oral') in ('mdi', 'pump', 'basal', 'intensive')
    icr = seed.get('icr', 12)        # insulin-to-carb ratio (g/unit)
    isf = seed.get('isf', 40)        # insulin sensitivity factor (mg/dL/unit)
    
    events = []
    glucose = prev_glucose
    
    # Fasting glucose (dawn phenomenon adds ~10–30 for T1)
    dawn = rng.randint(5, 25) if diabetes_type == 'Type 1' else rng.randint(0, 10)
    glucose = max(70, glucose + dawn + rng.randint(-15, 15))
    fasting_time = f"0{rng.randint(7,9)}:{rng.choice(['00','15','30','45'])}"
    events.append(f"{fasting_time} G={glucose}")
    
    # Breakfast
    meal, carbs = rng.choice(MEALS['breakfast'])
    btime = f"0{rng.randint(7,9)}:{rng.choice(['30','45'])}" if rng.random() > 0.1 else None
    if btime:
        insulin_str = ""
        if uses_insulin and rng.random() > 0.1:
            units = round(carbs / icr + rng.uniform(-0.5, 0.5), 1)
            units = max(0.5, units)
            insulin_str = f" I={units}R"
        events.append(f"{btime} M={meal} C={carbs}{insulin_str}")
        
        # Post-breakfast glucose (spike)
        spike = int(carbs * rng.uniform(2.5, 4.5))
        if uses_insulin:
            insulin_units = float(insulin_str.split('=')[1].replace('R','')) if insulin_str else 0
            spike -= int(insulin_units * isf * 0.6)  # partial absorption
        glucose_post = max(70, glucose + spike)
        post_time = f"{int(btime.split(':')[0]) + 2}:{btime.split(':')[1]}"
        label = " (Spike)" if glucose_post > 180 else ""
        events.append(f"{post_time} G={glucose_post}{label}")
        glucose = glucose_post
    
    # Optional exercise
    if rng.random() < 0.3:
        ex_time = f"{rng.randint(10,11)}:{rng.choice(['00','30'])}"
        ex_type = rng.choice(EXERCISES)
        glucose = max(70, glucose - rng.randint(15, 35))
        events.append(f"{ex_time} EX={ex_type} G={glucose}")
    
    # Lunch
    lunch_time = f"{rng.randint(12,13)}:{rng.choice(['00','30'])}"
    events.append(f"{lunch_time} G={max(70, glucose + rng.randint(-10, 20))}")
    meal, carbs = rng.choice(MEALS['lunch'])
    insulin_str = ""
    if uses_insulin and rng.random() > 0.08:
        units = round(carbs / icr + rng.uniform(-0.5, 0.5), 1)
        insulin_str = f" I={max(0.5, units)}R"
    events.append(f"{lunch_time} M={meal} C={carbs}{insulin_str}")
    glucose = max(70, glucose + int(carbs * 1.5) - (int(float(insulin_str.split('=')[1].replace('R','')) * isf * 0.4) if insulin_str else 0))
    
    # Afternoon glucose + optional snack/hypo treatment
    afternoon_time = f"{rng.randint(15,16)}:{rng.choice(['00','30'])}"
    glucose = max(65, glucose + rng.randint(-30, 10))
    hypo_label = " (Hypoglycemia)" if glucose < 70 else ""
    events.append(f"{afternoon_time} G={glucose}{hypo_label}")
    if glucose < 70:
        snack, carbs_rescue = rng.choice(MEALS['snack'])
        events.append(f"{afternoon_time} M={snack} C={carbs_rescue}")
        glucose = min(140, glucose + carbs_rescue * 3)
    
    # Dinner
    dinner_time = f"{rng.randint(18,20)}:{rng.choice(['00','30'])}"
    events.append(f"{dinner_time} G={glucose + rng.randint(-15, 15)}")
    meal, carbs = rng.choice(MEALS['dinner'])
    insulin_str = ""
    if uses_insulin and rng.random() > 0.08:
        units = round(carbs / icr + rng.uniform(-1.0, 1.0), 1)
        insulin_str = f" I={max(0.5, units)}R"
    events.append(f"{dinner_time} M={meal} C={carbs}{insulin_str}")
    
    # Long-acting insulin at bedtime (if applicable)
    basal_u = seed.get('basal_u', 0)
    if basal_u > 0:
        bedtime = f"2{rng.randint(1,3)}:{rng.choice(['00','30'])}"
        events.append(f"{bedtime} I={basal_u}L")
    
    # Bedtime glucose
    glucose_end = max(70, glucose + int(carbs * 1.8) - (int(float(insulin_str.split('=')[1].replace('R','')) * isf * 0.5) if insulin_str else 0))
    events.append(f"23:{rng.choice(['00','30'])} G={glucose_end}")
    
    log_line = f"D{day} " + " | ".join(events)
    return log_line, glucose_end


def generate_patient_log(seed: dict, n_days: int = 10) -> str:
    """Generate a full n-day log for a patient seed."""
    glucose = seed.get('fasting_glucose', 120) + random.randint(-15, 15)
    lines = []
    for day in range(1, n_days + 1):
        line, glucose = generate_daily_log(seed, day, glucose)
        lines.append(line)
    return '\n'.join(lines)


print("Log generator loaded.")

# Demo: generate a sample patient log
sample_seed = seeds[0] if seeds else {
    'seed_id': 1, 'diabetes_type': 'Type 1', 'regimen': 'mdi',
    'icr': 10, 'isf': 40, 'basal_u': 22, 'fasting_glucose': 128
}
sample_log = generate_patient_log(sample_seed, n_days=10)
print("\nSample 10-day log:")
print(sample_log)

## 3. Frontier Model Prompting

The function below shows the exact prompt structure used to query Claude/GPT-4 for expert annotations. The key design:

- **Strict JSON schema** — forces structured output
- **Few-shot examples** — 2 examples in the prompt for consistency
- **Grounding rule** — the model must never invent values not in the log
- **Batch prompting** — request N records per API call to maximize efficiency

In [ ]:
def build_batch_prompt(seeds_batch: list, category: str = 'analysis') -> str:
    """Build a prompt for the frontier model to annotate a batch of seeds.
    
    category: 'analysis' for pattern detection, 'advisory' for recommendations
    """
    schema = {
        'analysis': {
            'output_keys': '["feature", "patterns", "message"]',
            'description': 'dominant glycemic pattern with evidence from logs',
        },
        'advisory': {
            'output_keys': '["meal_recommendations", "icr_assessment", "lifestyle_advice"]',
            'description': 'personalised management recommendations',
        }
    }[category]
    
    examples_str = "\n".join([
        json.dumps({
            'profile': f"Age: {s.get('age', 45)}, Sex: {s.get('sex', 'M')}, "
                       f"Diabetes Type: {s.get('diabetes_type', 'Type 2')}, "
                       f"Weight: {s.get('weight_kg', 85)} kg",
            'input': generate_patient_log(s, n_days=10),
            'reasoning': '<FILL: detailed step-by-step clinical reasoning>',
            'output': f'<FILL: valid JSON with keys {schema["output_keys"]}>',
            'category': category,
            'diabetes_type': s.get('diabetes_type', 'Type 2')
        }, ensure_ascii=False)
        for s in seeds_batch
    ])
    
    prompt = f"""You are a clinical diabetes specialist.
For each patient below, analyse their 10-day glucose log and produce a structured annotation.

STRICT RULES:
1. Base ALL observations ONLY on values explicitly present in the logs.
2. Never invent glucose values, meal frequencies, or patterns not visible in the data.
3. The 'reasoning' field must be ≥150 characters of step-by-step clinical thinking.
4. The 'output' field must be a valid JSON string with keys: {schema['output_keys']}.
5. Output each record as a single JSON object on one line (no pretty printing).

Produce {len(seeds_batch)} JSON records, one per line:

{examples_str}"""
    
    return prompt


# Show sample prompt (first 2 seeds)
if len(seeds) >= 2:
    sample_prompt = build_batch_prompt(seeds[:2], category='analysis')
    print(f"Sample prompt (first 500 chars):")
    print(sample_prompt[:500])
    print(f"\n... (total: {len(sample_prompt)} chars)")
else:
    print("Load master_seeds.json to see sample prompt.")

## 4. Batch Processing Summary

The actual batch generation was done manually (sending prompts to Claude/GPT-4 via API or UI) and the raw responses were saved to `../step2_raw_batches/batch_01.txt` through `batch_42.txt`.

**Processing history:**
- Batches 1–12: Initial generation, analysis category, Type 1 focus
- Batches 13–24: Expanded to Type 2, added advisory category
- Batches 25–30: Added exercise scenarios (EX= events)
- Batches 31–36: Edge cases (dawn phenomenon, fat-delayed spikes)
- Batches 37–42: Rebalancing underrepresented patterns

Total output: 42 batches × ~30 examples = ~1,260 raw records before parsing.

In [ ]:
# Inspect the raw batch files
import os

raw_dir = '../step2_raw_batches'
batch_files = sorted(f for f in os.listdir(raw_dir) if f.endswith('.txt'))
print(f"Total batch files: {len(batch_files)}")

# Count lines per batch
for fname in batch_files[:5]:
    path = os.path.join(raw_dir, fname)
    with open(path, 'r', encoding='utf-8') as f:
        lines = [l for l in f.readlines() if l.strip()]
    print(f"  {fname}: {len(lines)} lines")

print("  ... (showing first 5)")

## 5. Next Steps

The raw batch files are now ready for parsing:

```bash
cd ../step3_data_processing
python parser.py
```

This will:
- Fix JSON formatting issues
- Validate all required fields and content quality
- Remove duplicates
- Output the clean `diabetix_training_data.jsonl` to `../step4_training_dataset/`

See `../step3_data_processing/02_data_processing_analysis.ipynb` for exploration of the processed dataset.